In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("tc_with_composition_parsed.csv")
df

,Formula,Temperature (K),TC,Pt,Ni,Al,Si,Fe,Cu,Zn,...,Bi,La,Li,Na,Te,Ca,Gd,Ta,Hf,P
0,Pt,285.66,77.04,100.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.00,0
1,Pt,560.50,85.13,100.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.00,0
2,Pt,678.34,87.32,100.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.00,0
3,Pt,783.08,89.23,100.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.00,0
4,Pt,891.12,90.36,100.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6254,Hf30.71Ti8.24Ni20.20Sn40.85,421.37,2.35,0.0,20.2,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,30.71,0
6255,Hf30.71Ti8.24Ni20.20Sn40.85,474.69,2.32,0.0,20.2,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,30.71,0
6256,Hf30.71Ti8.24Ni20.20Sn40.85,521.51,2.30,0.0,20.2,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,30.71,0
6257,Hf30.71Ti8.24Ni20.20Sn40.85,576.04,2.30,0.0,20.2,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,30.71,0


In [3]:

# Cột target
target_col = "TC"

# X = tất cả các cột trừ target_col
X = df.drop(columns=[target_col, "Formula"])

# y = cột target
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (6259, 50)
y shape: (6259,)


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 7)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (5007, 50)
Test: (1252, 50)


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


In [6]:

cv = KFold(n_splits=5, shuffle=True, random_state=42)

models_and_grids = {

    "DecisionTree": (
        DecisionTreeRegressor(random_state=42),
        {
            "max_depth": [None, 6, 12, 20],
            "min_samples_split": [2, 5, 10]
        }
    ),
    "RandomForest": (
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            "n_estimators": [200, 500],
            "max_depth": [None, 12, 20],
            "min_samples_split": [2, 5]
        }
    ),
    "ExtraTrees": (
        ExtraTreesRegressor(random_state=42, n_jobs=-1),
        {
            "n_estimators": [300, 600],
            "max_depth": [None, 12, 20],
            "min_samples_split": [2, 5]
        }
    ),
    "GradientBoosting": (
        GradientBoostingRegressor(random_state=42),
        {
            "n_estimators": [200, 500],
            "learning_rate": [0.05, 0.1],
            "max_depth": [2, 3, 4]
        }
    ),
    
    "XGBoost": (
        XGBRegressor(
            random_state=42, n_estimators=300, tree_method="hist", n_jobs=-1,
            eval_metric="rmse"
        ),
        {
            "max_depth": [3, 5, 7],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0]
        }
    ),
    "CatBoost": (
        CatBoostRegressor(
            random_state=42, silent=True, loss_function="RMSE"
        ),
        {
            "iterations": [500, 800],
            "depth": [4, 6, 8],
            "learning_rate": [0.05, 0.1],
            "l2_leaf_reg": [1, 3, 5]
        }
    ),
}



In [7]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

results = []

# Hàm tính R2, RMSE, MAE đúng chuẩn
def r2_rmse_mae(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return r2, rmse, mae

for name, (estimator, grid) in models_and_grids.items():
    print(f"\n===== {name} =====")

    gs = GridSearchCV(
        estimator=estimator,
        param_grid=grid,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=0
    )
    gs.fit(X_train, y_train)

    best = gs.best_estimator_

    # Predict
    y_pred_tr  = best.predict(X_train)
    y_pred_te  = best.predict(X_test)

    # Metrics
    r2_tr,  rmse_tr,  mae_tr  = r2_rmse_mae(y_train, y_pred_tr)
    r2_te,  rmse_te,  mae_te  = r2_rmse_mae(y_test, y_pred_te)

    # Lưu kết quả
    results.append({
        "model": name,
        "best_params": gs.best_params_,

        "R2_train": r2_tr,   "RMSE_train": rmse_tr,   "MAE_train": mae_tr,
        "R2_test": r2_te,    "RMSE_test": rmse_te,    "MAE_test": mae_te
    })

# Kết quả DataFrame
res_df = pd.DataFrame(results).sort_values("RMSE_test")
res_df



===== DecisionTree =====

===== RandomForest =====

===== ExtraTrees =====

===== GradientBoosting =====

===== XGBoost =====

===== CatBoost =====


,model,best_params,R2_train,RMSE_train,MAE_train,R2_test,RMSE_test,MAE_test
2,ExtraTrees,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.999934,0.733935,0.026727,0.987739,10.469776,2.283805
5,CatBoost,"{'depth': 8, 'iterations': 800, 'l2_leaf_reg':...",0.998977,2.887610,1.791594,0.983521,12.137936,3.495803
4,XGBoost,"{'colsample_bytree': 0.8, 'learning_rate': 0.1...",0.998261,3.765227,1.990640,0.981099,12.999230,3.978008
1,RandomForest,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.997894,4.142849,1.366909,0.974512,15.095373,4.496660
3,GradientBoosting,"{'learning_rate': 0.1, 'max_depth': 4, 'n_esti...",0.992234,7.956598,4.487763,0.974370,15.137376,6.512272
0,DecisionTree,"{'max_depth': None, 'min_samples_split': 2}",0.999934,0.733906,0.025980,0.964479,17.820510,6.569387


In [8]:
feature_columns = X.columns.tolist()
print(feature_columns)


['Temperature (K)', 'Pt', 'Ni', 'Al', 'Si', 'Fe', 'Cu', 'Zn', 'Mn', 'Mg', 'Pd', 'Pb', 'Au', 'Ag', 'Co', 'N', 'I', 'Ti', 'Sr', 'Cr', 'Zr', 'V', 'Mo', 'C', 'O', 'Ce', 'Ga', 'Ge', 'As', 'Ru', 'Rh', 'Sn', 'Sb', 'Re', 'Os', 'Ir', 'S', 'Se', 'W', 'Y', 'Bi', 'La', 'Li', 'Na', 'Te', 'Ca', 'Gd', 'Ta', 'Hf', 'P']


In [9]:
# Lấy best model (model có RMSE_val nhỏ nhất)

best_row = res_df.iloc[0]
best_model_name = best_row["model"]
print(f"Best Model: {best_model_name}")
print("Best Params:", best_row["best_params"])

# Lấy class của model (CatBoostRegressor, RandomForestRegressor, v.v.)
model_class = type(models_and_grids[best_model_name][0])

# Khởi tạo model mới với best params
best_estimator = model_class(**best_row["best_params"])

# Fit lại model
best_estimator.fit(X_train, y_train)

Best Model: ExtraTrees
Best Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}


ExtraTreesRegressor(n_estimators=300)

In [12]:
import joblib

joblib.dump(best_estimator, "best_estimator_extratrees_TC_alloys.plk")


['best_estimator_extratrees_TC_alloys.plk']

In [13]:
import joblib

best_ET_model = joblib.load("best_estimator_extratrees_TC_alloys.plk")
